In [ ]:
"""
Create a publication-style forest plot with readable variable labels.

Expected CSV columns:
    Feature, OR, CI_low, CI_high

Optional columns:
    p_value, p_value_fdr, Error

Run:
    python forest_plot_clean_labels.py "your_results.csv"

Outputs:
    forest_plot_clean_labels.png
    forest_plot_clean_labels.pdf
    results_with_clean_variable_names.csv
"""

from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


LABEL_MAP = {
    "Depression": "Depression symptoms",
    "SelfEsteem": "Self-esteem issues",
    "FEMOTION": "Emotional symptoms",
    "FPEER": "Peer problems",
    "child_ilness_Yes": "Childhood illness: yes",
    "sex_male": "Male sex",
    "mKessler": "Parent 1 psychological distress",
    "BMI": "BMI",
    "mDepression_Yes": "Parent 1 depression: yes",
    "Obesity_obese": "Weight status: obese",
    "child_alcohol_many": "Alcohol use: many occasions",
    "mExtravert": "Parent 1 extraversion",
    "mNeurotic": "Parent 1 neuroticism",
    "child_alcohol_some": "Alcohol use: some occasions",
    "cog_risk_taking": "Risk-taking",
    "knockedout_Yes": "History of loss of consciousness: yes",
    "mReligion_hindu": "Parent 1 religion: Hindu",
    "fExtravert": "Parent 2 extraversion",
    "child_cannabis_more than 5": "Cannabis use: more than 5 times",
    "mEthnicity_non-white": "Parent 1 ethnicity: non-White",
    "FCONDUCT": "Conduct problems",
    "mEdu_higher edu": "Parent 1 education: higher education",
    "mConscienc": "Parent 1 conscientiousness",
    "cog_risk_adjustment": "Risk adjustment",
    "mEmploy_management": "Parent 1 employment: managerial",
    "mOpenness": "Parent 1 openness",
    "Obesity_overweight": "Weight status: overweight",
    "child_cannabis_one to four": "Cannabis use: 1–4 times",
    "mReligion_muslim": "Parent 1 religion: Muslim",
    "FPROSOC": "Prosocial behaviour",
    "fOpenness": "Parent 2 openness",
    "fAlcohol_binary_high risk": "Parent 2 alcohol use: high risk",
    "fDepression_Yes": "Parent 2 depression: yes",
    "banghead_Yes": "History of head injury: yes",
    "mean_acc_24h": "Mean 24-hour acceleration",
    "mvpa_acc_5sec": "Moderate-to-vigorous physical activity",
    "fKessler": "Parent 2 psychological distress",
    "mAlcohol_binary_high risk": "Parent 1 alcohol use: high risk",
    "mEdu_overseas": "Parent 1 education: overseas qualification",
    "fAgree": "Parent 2 agreeableness",
    "m5_hour_start": "Most active 5 hour block start time",
    "mEmploy_s-emp": "Parent 1 employment: self-employed",
    "fConscienc": "Parent 2 conscientiousness",
    "cog_decision_making": "Decision-making",
    "sexualassault_Yes": "History of sexual assault: yes",
    "fNeurotic": "Parent 2 neuroticism",
    "mAgree": "Parent 1 agreeableness",
    "fAlcohol": "Parent 2 alcohol use",
    "child_specneeds_Yes": "Special educational needs: yes",
    "child_autism_Yes": "Autism diagnosis: yes",
    "mEmploy_semi-rou and routine": "Parent 1 employment: semi-routine or routine",
    "cog_word_activity": "Word activity score",
    "FHYPER": "Hyperactivity/inattention",
    "l5_hour_start": "Least active 5h block start time",
    "child_ADHD_Yes": "ADHD diagnosis: yes",
    "mReligion_none": "Parent 1 religion: none",
    "mEmploy_lo sup and tech": "Parent 1 employment: lower supervisory or technical",
    "mReligion_christian": "Parent 1 religion: Christian",
    "mAlcohol": "Parent 1 alcohol use",
    "mAge": "Parent 1 age",
    "mReligion_sikh": "Parent 1 religion: Sikh",
    "mReligion_other": "Parent 1 religion: other",
}


def make_forest_plot(
    csv_file,
    output_png="forest_plot_clean_labels.png",
    output_pdf="forest_plot_clean_labels.pdf",
    output_clean_csv="results_with_clean_variable_names.csv",
    use_fdr=True,
    significant_only=False,
    sort_by="OR",
    title="Multivariable logistic regression",
):
    df = pd.read_csv(csv_file)

    required = {"Feature", "OR", "CI_low", "CI_high"}
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(
            f"Missing required column(s): {sorted(missing)}. "
            f"Available columns: {list(df.columns)}"
        )

    # Replace raw feature names with clean manuscript labels.
    df["Feature_raw"] = df["Feature"]
    df["Feature"] = df["Feature"].map(LABEL_MAP).fillna(df["Feature"])

    # Save a cleaned results table before plotting.
    df.to_csv(output_clean_csv, index=False)

    for col in ["OR", "CI_low", "CI_high"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Remove estimates that could not be calculated.
    df = df.dropna(subset=["Feature", "OR", "CI_low", "CI_high"]).copy()
    df = df[(df["OR"] > 0) & (df["CI_low"] > 0)].copy()

    finite_high = df.loc[np.isfinite(df["CI_high"]), "CI_high"]
    if finite_high.empty:
        raise ValueError("No finite upper confidence intervals were found.")

    # Cap infinite CIs for display and mark them with arrows.
    ci_cap = max(10.0, float(finite_high.max()) * 1.15)
    df["CI_high_infinite"] = ~np.isfinite(df["CI_high"])
    df["CI_high_plot"] = df["CI_high"].clip(upper=ci_cap)

    p_col = None
    if use_fdr and "p_value_fdr" in df.columns:
        p_col = "p_value_fdr"
    elif "p_value" in df.columns:
        p_col = "p_value"

    if p_col is not None:
        df[p_col] = pd.to_numeric(df[p_col], errors="coerce")
        df["significant"] = df[p_col] < 0.05
    else:
        df["significant"] = False

    if significant_only:
        df = df[df["significant"]].copy()

    if df.empty:
        raise ValueError("No valid rows remain after filtering.")

    if sort_by == "Feature":
        df = df.sort_values("Feature", ascending=False)
    elif sort_by in {"p_value", "p_value_fdr"} and sort_by in df.columns:
        df = df.sort_values(sort_by, ascending=False)
    else:
        df = df.sort_values("OR", ascending=False)

    y = np.arange(len(df))
    lower_error = df["OR"] - df["CI_low"]
    upper_error = df["CI_high_plot"] - df["OR"]

    fig_height = max(6, 0.38 * len(df) + 1.8)
    fig, ax = plt.subplots(figsize=(11, fig_height))

    for is_sig, marker, label in [
        (False, "o", "Not significant"),
        (True, "s", "FDR-adjusted p < .05"),
    ]:
        mask = df["significant"] == is_sig
        if not mask.any():
            continue

        ax.errorbar(
            df.loc[mask, "OR"],
            y[mask],
            xerr=np.vstack([
                lower_error.loc[mask],
                upper_error.loc[mask],
            ]),
            fmt=marker,
            linestyle="none",
            capsize=3,
            markersize=5,
            label=label,
        )

    for row_position, (_, row) in enumerate(df.iterrows()):
        if row["CI_high_infinite"]:
            ax.annotate(
                "",
                xy=(ci_cap, row_position),
                xytext=(ci_cap / 1.35, row_position),
                arrowprops={"arrowstyle": "->"},
            )

    ax.axvline(1, linestyle="--", linewidth=1)
    ax.set_xscale("log")
    ax.set_yticks(y)
    ax.set_yticklabels(df["Feature"])
    ax.set_xlabel("Odds ratio (95% CI: logarithmic scale)")
    ax.set_title(title)
    ax.grid(axis="x", linestyle=":", alpha=0.5)

    for row_position, (_, row) in enumerate(df.iterrows()):
        upper_text = "∞" if row["CI_high_infinite"] else f"{row['CI_high']:.2f}"
        estimate_text = (
            f"{row['OR']:.2f} [{row['CI_low']:.2f}, {upper_text}]"
        )
        ax.text(
            1.01,
            row_position,
            estimate_text,
            transform=ax.get_yaxis_transform(),
            va="center",
            fontsize=8,
        )

    ax.text(
        1.01,
        1.01,
        "OR [95% CI]",
        transform=ax.transAxes,
        va="bottom",
        fontsize=9,
        fontweight="bold",
    )

    if p_col is not None:
        ax.legend(loc="lower right")

    fig.subplots_adjust(left=0.38, right=0.79, top=0.97, bottom=0.05)
    fig.savefig(output_png, dpi=300, bbox_inches="tight")
    fig.savefig(output_pdf, bbox_inches="tight")
    plt.close(fig)

    return df

csv_path = Path("univariate_odds_ratios_95CI_actigraphy_only_ALL.csv")

make_forest_plot(

    csv_file=csv_path,

    output_png="forest_plot_clean_labels.png",

    output_pdf="forest_plot_clean_labels.pdf",

    output_clean_csv="results_with_clean_variable_names.csv",

    use_fdr=True,

    significant_only=False,

    sort_by="OR",

    title="Univariate logistic regression",

)

print("Done!")
